## Gold — KPI Network

Calculates network KPIs per city and transport type from silver tables.

### Source
- `gtfs_silver.routes` — route list with transport type
- `gtfs_silver.trips` — trips linked to routes and services
- `gtfs_silver.stop_times` — stops served per trip
- `gtfs_silver.calendar_dates` — active service dates

### Output
`gtfs_gold.kpi_network`

| Column | Description |
|--------|-------------|
| city | City name |
| route_type_desc | Transport type (bus, metro, tram, ...) |
| total_routes | Total distinct routes |
| total_stops_served | Total stops served across all routes |
| avg_stops_per_route | Average stops per route |
| total_trips_weekday | Distinct trips active on at least one weekday |
| total_trips_weekend | Distinct trips active on at least one weekend day |

### CTE Logic
- **stops_per_route** — joins `trips` → `stop_times` to count distinct stops per route
- **weekday_trips** — joins `trips` → `calendar_dates` filtering weekdays (DAYOFWEEK 2-6)
- **weekend_trips** — same, filtering saturday (7) and sunday (1)

In [0]:
CREATE OR REPLACE TABLE gtfs_gold.kpi_network AS
WITH stops_per_route AS (
    SELECT 
        t.route_id,
        t.city,
        COUNT(DISTINCT st.stop_id) AS n_stops
    FROM gtfs_silver.trips t
    JOIN gtfs_silver.stop_times st ON t.trip_id = st.trip_id AND t.city=st.city
    GROUP BY t.route_id, t.city
),
weekday_trips AS (
    SELECT
        t.route_id,
        t.city,
        t.trip_id
    FROM gtfs_silver.trips t
    JOIN gtfs_silver.calendar_dates cd ON t.service_id = cd.service_id AND t.city=cd.city
    WHERE DAYOFWEEK(cd.date) BETWEEN 2 AND 6
),
weekend_trips AS (
    SELECT
        t.route_id,
        t.city,
        t.trip_id
    FROM gtfs_silver.trips t
    JOIN gtfs_silver.calendar_dates cd ON t.service_id = cd.service_id AND t.city=cd.city
    WHERE DAYOFWEEK(cd.date) IN (1,7)
)
SELECT
    r.city,
    r.route_type_desc,
    COUNT(DISTINCT r.route_id)        AS total_routes,
    SUM(spr.n_stops)                  AS total_stops_served,
    ROUND(AVG(spr.n_stops), 1)        AS avg_stops_per_route,
    COUNT(DISTINCT wdt.trip_id)       AS total_trips_weekday,
    COUNT(DISTINCT wet.trip_id)       AS total_trips_weekend
FROM gtfs_silver.routes r
LEFT JOIN stops_per_route spr ON r.route_id=spr.route_id AND r.city=spr.city
LEFT JOIN weekday_trips wdt ON r.route_id=wdt.route_id AND r.city=wdt.city
LEFT JOIN weekend_trips wet ON r.route_id=wet.route_id AND r.city=wet.city
GROUP BY r.city, r.route_type_desc;

In [0]:
SELECT city, route_type_desc, total_routes, total_trips_weekday, total_trips_weekend
FROM gtfs_gold.kpi_network
ORDER BY city, total_trips_weekday DESC